# FHIA — Bloque 13  
## Práctica: Redes neuronales en acción

**Objetivo:** entrenar redes neuronales simples sobre datasets sintéticos para observar:

- cómo aparecen fronteras no lineales,
- qué aportan las capas ocultas,
- cómo impactan las funciones de activación,
- cómo se detecta el overfitting,
- cómo ayuda la regularización.

> La práctica no busca implementar backpropagation desde cero.  
> El objetivo es interpretar cómo aprende una red neuronal.

## 0. Instrucciones generales

Ejecutá el notebook de arriba hacia abajo.

En varias secciones vas a encontrar bloques llamados **Actividad**.  
Ahí tenés que modificar hiperparámetros, volver a entrenar y responder preguntas conceptuales.

No se evalúa la calidad del código.  
Se evalúa la interpretación.

## 1. Imports

Usaremos `scikit-learn` para crear datasets sintéticos y entrenar redes simples con `MLPClassifier`.

Esto nos permite concentrarnos en el comportamiento de la red, no en detalles de bajo nivel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

np.random.seed(42)

## 2. Funciones auxiliares

Estas funciones ya están dadas.

Sirven para:

- graficar datos,
- entrenar modelos,
- visualizar fronteras de decisión,
- comparar desempeño en train y test.

In [ ]:
def plot_dataset(X, y, title="Dataset"):
    plt.figure(figsize=(6, 5))
    plt.scatter(X[:, 0], X[:, 1], c=y, s=35, edgecolor="k", alpha=0.85)
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()


def plot_decision_boundary(model, X, y, title="Frontera de decisión"):
    x_min, x_max = X[:, 0].min() - 0.7, X[:, 0].max() + 0.7
    y_min, y_max = X[:, 1].min() - 0.7, X[:, 1].max() + 0.7

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 350),
        np.linspace(y_min, y_max, 350)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, Z, alpha=0.30)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=35, edgecolor="k", alpha=0.85)
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()


def train_mlp(
    X_train, y_train,
    hidden_layer_sizes=(8, 8),
    activation="relu",
    alpha=0.0001,
    learning_rate_init=0.01,
    max_iter=1000,
    random_state=42
):
    model = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation=activation,
            alpha=alpha,
            learning_rate_init=learning_rate_init,
            max_iter=max_iter,
            random_state=random_state
        )
    )
    model.fit(X_train, y_train)
    return model


def evaluate_model(model, X_train, y_train, X_test, y_test):
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"Train accuracy: {train_acc:.3f}")
    print(f"Test accuracy:  {test_acc:.3f}")
    print(f"Gap train-test: {train_acc - test_acc:.3f}")
    return train_acc, test_acc


def plot_loss_curve(model, title="Curva de pérdida"):
    mlp = model.named_steps["mlpclassifier"]
    plt.figure(figsize=(6, 4))
    plt.plot(mlp.loss_curve_)
    plt.title(title)
    plt.xlabel("Épocas")
    plt.ylabel("Loss")
    plt.show()

## 3. Dataset: `moons`

Vamos a empezar con un dataset sintético no lineal.

La ventaja de estos datasets es que permiten ver claramente la frontera de decisión.

In [ ]:
X, y = make_moons(
    n_samples=500,
    noise=0.25,
    random_state=42
)

plot_dataset(X, y, "Dataset moons: problema no lineal")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

## 4. Modelo lineal: una frontera no alcanza

Antes de entrenar una red neuronal, probemos un clasificador lineal.

Esto simula la situación conceptual del perceptrón: una frontera lineal sobre un problema que no es lineal.

In [ ]:
linear_model = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)

linear_model.fit(X_train, y_train)

evaluate_model(linear_model, X_train, y_train, X_test, y_test)
plot_decision_boundary(linear_model, X, y, "Modelo lineal: frontera insuficiente")

### Preguntas

1. ¿La frontera lineal captura bien la estructura del dataset?
2. ¿Qué tipo de errores comete?
3. ¿Por qué este ejemplo se parece conceptualmente al problema XOR?

## 5. Red neuronal simple

Ahora entrenamos una red con capas ocultas.

La idea es observar cómo cambia la frontera de decisión cuando agregamos capacidad de representación.

In [ ]:
mlp = train_mlp(
    X_train, y_train,
    hidden_layer_sizes=(8, 8),
    activation="relu",
    alpha=0.0001,
    learning_rate_init=0.01,
    max_iter=1000,
    random_state=42
)

evaluate_model(mlp, X_train, y_train, X_test, y_test)
plot_decision_boundary(mlp, X, y, "MLP con capas ocultas: frontera no lineal")
plot_loss_curve(mlp, "Curva de pérdida — MLP")

### Preguntas

1. ¿Qué cambió respecto del modelo lineal?
2. ¿La frontera parece más flexible?
3. ¿Dirías que la red está memorizando o aprendiendo un patrón general?
4. ¿Dónde ves la idea de “representación aprendida”?

## 6. Actividad 1 — Cambiar arquitectura

Modificá `hidden_layer_sizes` y compará resultados.

Probá al menos estas arquitecturas:

- `(4,)`
- `(8, 8)`
- `(32, 32, 32)`
- `(64, 64, 64, 64)`

Observá:

- accuracy en train,
- accuracy en test,
- forma de la frontera,
- diferencia entre train y test.

In [ ]:
# ACTIVIDAD 1
# Cambiá la arquitectura y volvé a ejecutar esta celda.

arquitectura = (8, 8)  # probar: (4,), (8, 8), (32, 32, 32), (64, 64, 64, 64)

model_arch = train_mlp(
    X_train, y_train,
    hidden_layer_sizes=arquitectura,
    activation="relu",
    alpha=0.0001,
    learning_rate_init=0.01,
    max_iter=1000,
    random_state=42
)

evaluate_model(model_arch, X_train, y_train, X_test, y_test)
plot_decision_boundary(model_arch, X, y, f"Arquitectura: {arquitectura}")
plot_loss_curve(model_arch, f"Curva de pérdida — Arquitectura {arquitectura}")

### Responder

1. ¿Qué arquitectura produjo la frontera más simple?
2. ¿Qué arquitectura produjo la frontera más compleja?
3. ¿Más neuronas siempre mejoran la generalización?
4. ¿Qué señales mirarías para sospechar overfitting?

## 7. Actividad 2 — Cambiar función de activación

Ahora probamos distintas activaciones.

En `MLPClassifier`, las opciones principales son:

- `relu`
- `tanh`
- `logistic`  (sigmoid)

Compará velocidad de entrenamiento, loss final y frontera.

In [ ]:
# ACTIVIDAD 2
# Cambiá la activación y volvé a ejecutar.

activacion = "relu"  # probar: "relu", "tanh", "logistic"

model_act = train_mlp(
    X_train, y_train,
    hidden_layer_sizes=(8, 8),
    activation=activacion,
    alpha=0.0001,
    learning_rate_init=0.01,
    max_iter=1500,
    random_state=42
)

evaluate_model(model_act, X_train, y_train, X_test, y_test)
plot_decision_boundary(model_act, X, y, f"Activación: {activacion}")
plot_loss_curve(model_act, f"Curva de pérdida — Activación {activacion}")

### Responder

1. ¿Qué activación produjo mejor resultado?
2. ¿Cuál pareció entrenar más lento?
3. ¿Por qué una activación no lineal es necesaria?
4. ¿Qué pasaría si todas las capas fueran lineales?

## 8. Actividad 3 — Learning rate

El learning rate controla el tamaño del paso en gradiente descendente.

Vamos a comparar:

- demasiado chico,
- razonable,
- demasiado grande.

In [ ]:
# ACTIVIDAD 3
# Probá distintos learning rates.

lr = 0.01  # probar: 0.00001, 0.01, 1.0, 10.0

model_lr = train_mlp(
    X_train, y_train,
    hidden_layer_sizes=(8, 8),
    activation="relu",
    alpha=0.0001,
    learning_rate_init=lr,
    max_iter=1000,
    random_state=42
)

evaluate_model(model_lr, X_train, y_train, X_test, y_test)
plot_decision_boundary(model_lr, X, y, f"Learning rate: {lr}")
plot_loss_curve(model_lr, f"Curva de pérdida — Learning rate {lr}")

### Responder

1. ¿Qué ocurre con learning rate muy pequeño?
2. ¿Qué ocurre con learning rate muy grande?
3. ¿El learning rate cambia la capacidad de representación o la optimización?
4. ¿Cómo se conecta esto con la metáfora de “bajar la montaña”?

## 9. Provocar overfitting

Ahora vamos a crear una situación propicia para overfitting:

- pocos datos de entrenamiento,
- red grande,
- muchas iteraciones,
- poca regularización.

El objetivo es ver que una red puede ajustar demasiado bien el entrenamiento y generalizar peor.

In [ ]:
# Dataset más pequeño para facilitar overfitting
X_small, y_small = make_moons(
    n_samples=120,
    noise=0.30,
    random_state=7
)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_small, y_small,
    test_size=0.50,
    random_state=42,
    stratify=y_small
)

plot_dataset(X_small, y_small, "Dataset pequeño y ruidoso")

overfit_model = train_mlp(
    X_train_s, y_train_s,
    hidden_layer_sizes=(64, 64, 64),
    activation="relu",
    alpha=0.000001,
    learning_rate_init=0.01,
    max_iter=3000,
    random_state=42
)

evaluate_model(overfit_model, X_train_s, y_train_s, X_test_s, y_test_s)
plot_decision_boundary(overfit_model, X_small, y_small, "Red grande con pocos datos: posible overfitting")
plot_loss_curve(overfit_model, "Curva de pérdida — Modelo sobrecapacitado")

### Preguntas

1. ¿La frontera parece más irregular?
2. ¿Hay diferencia entre accuracy de train y test?
3. ¿Qué evidencia sugiere overfitting?
4. ¿Qué relación hay entre capacidad del modelo y riesgo de memorización?

## 10. Regularización L2

Ahora repetimos el experimento anterior, pero aumentando `alpha`.

En `MLPClassifier`, `alpha` controla la regularización L2.

Valores más altos penalizan pesos grandes y tienden a suavizar la frontera.

In [ ]:
regularized_model = train_mlp(
    X_train_s, y_train_s,
    hidden_layer_sizes=(64, 64, 64),
    activation="relu",
    alpha=0.1,  # regularización L2 más fuerte
    learning_rate_init=0.01,
    max_iter=3000,
    random_state=42
)

evaluate_model(regularized_model, X_train_s, y_train_s, X_test_s, y_test_s)
plot_decision_boundary(regularized_model, X_small, y_small, "Modelo regularizado con L2")
plot_loss_curve(regularized_model, "Curva de pérdida — Modelo regularizado")

### Preguntas

1. ¿La frontera se volvió más suave?
2. ¿Cambió la diferencia entre train y test?
3. ¿La regularización mejoró la generalización?
4. ¿Por qué penalizar pesos grandes puede ayudar?

## 11. Comparación final: baja capacidad, alta capacidad, regularización

Ejecutá esta celda para comparar tres modelos:

1. modelo simple,
2. modelo grande,
3. modelo grande regularizado.

In [ ]:
models = {
    "Simple (4 neuronas)": train_mlp(
        X_train_s, y_train_s,
        hidden_layer_sizes=(4,),
        activation="relu",
        alpha=0.0001,
        learning_rate_init=0.01,
        max_iter=3000,
        random_state=42
    ),
    "Grande sin regularización": overfit_model,
    "Grande con L2": regularized_model
}

for name, model in models.items():
    print("\n" + name)
    evaluate_model(model, X_train_s, y_train_s, X_test_s, y_test_s)
    plot_decision_boundary(model, X_small, y_small, name)

## 12. Reflexión final

Responder en texto breve.

1. ¿Por qué un modelo lineal no alcanza para `moons`?
2. ¿Qué aportan las capas ocultas?
3. ¿Qué rol cumple la función de activación?
4. ¿Qué diferencia hay entre capacidad de representación y optimización?
5. ¿Cómo identificaste overfitting?
6. ¿Qué efecto tuvo la regularización?
7. ¿Qué significa que una red “aprenda una representación”?
8. ¿Qué conexión ves entre esta práctica y TensorFlow Playground?

## Idea clave

> Una red neuronal aprende ajustando pesos para transformar el espacio de entrada en una representación donde la tarea se vuelve más simple.

Pero aprender no es memorizar.  
El objetivo final es generalizar.